![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/language-detection/Language_Detection_with_GlotLID.ipynb)

# Find and remove non-English sentences with `LanguageDetectorDL` and GlotLID v4

Many collections are mostly English with some text in other languages, for example clinical notes that contain a sentence in Chinese or Spanish. When the next step only handles English (de-identification, an English NER model), the other-language text has to be found and set aside first.

The approach here:

1. Split each document into sentences, so one foreign sentence inside an English paragraph is not hidden by the paragraph's English majority.
2. Detect the language of each sentence with [GlotLID v4](https://huggingface.co/cis-lmu/glotlid) (`ld_glotlid_v4`, 2,060 labels such as `eng_Latn`, `cmn_Hani`, `spa_Latn`).
3. Keep English, the uncertain results (`unk`) and the non-language labels (`zxx_Code`, `zxx_Zmth`, `zxx_Zsym`: code, maths, symbols); set aside the rest.

## Setup

In [ ]:
!pip install -q pyspark==3.5.4 spark-nlp==7.0.1

In [ ]:
import sparknlp
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from sparknlp.annotator import LanguageDetectorDL, SentenceDetector
from sparknlp.base import DocumentAssembler

spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())

## The pipeline

- `SentenceDetector` splits sentences. The Chinese, Japanese and full-width marks `。！？` are added to its usual rules, because a Chinese sentence is often followed by English with no other boundary; `SentenceDetectorDLModel` (`sentence_detector_dl`, `xx`) can miss the `。` in front of English text and hand both languages over as one sentence.
- `LanguageDetectorDL.pretrained("ld_glotlid_v4", "xx")` comes with these settings, which you can change:
  - `threshold` 0.7 with `thresholdLabel` `unk`: below 0.7 the result is `unk`
  - `scriptPooling`: for a non-Latin top label the score is the sum of all labels in its script, so Chinese split between Mandarin and Cantonese still counts as confident
  - `minLetters` 20: when the top label is in Latin script, text with fewer than 20 letters becomes `unk`
  - `topK` 5: the metadata keeps the 5 best scores and the `confidence` the decision was based on
  - `coalesceSentences` false: one result per sentence
- `ld_glotlid_v4_int8` is a 196 MB version of the same model (the default is 782 MB).

In [ ]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("sentence") \
    .setCustomBounds(["。", "！", "？"]) \
    .setUseCustomBoundsOnly(False) \
    .setCustomBoundsStrategy("append")

language_detector = LanguageDetectorDL.pretrained("ld_glotlid_v4", "xx") \
    .setInputCols(["sentence"]) \
    .setOutputCol("language")

pipeline = Pipeline(stages=[document_assembler, sentence_detector, language_detector])

In [ ]:
notes = [
    (1, "HISTORY OF PRESENT ILLNESS: The patient is a 67-year-old man admitted with chest pain. "
        "He has a history of type 2 diabetes and hypertension. Meds: lisinopril 10mg PO daily, ASA 81mg daily."),
    (2, "The patient was seen with her daughter, who interpreted. 患者因胸痛和呼吸困难三天入院。 "
        "Vital signs were stable and she was discharged home."),
    (3, "El paciente refiere dolor torácico desde hace tres días. The physical exam showed mild edema. "
        "HbA1c 8.2%, LDL 131. No oral lesions."),
    (4, "He came with his son 张伟, who interpreted. Plan: follow up in 2 weeks."),
]
data = spark.createDataFrame(notes, ["id", "text"])
result = pipeline.fit(data).transform(data)

One row per sentence, with its language and the score the decision was based on:

In [ ]:
# a language annotation carries the index of its sentence, so the two are joined on it
# (an empty sentence gets no language annotation)
sentence_rows = result.select("id", F.posexplode("sentence.result").alias("n", "sentence"))
language_rows = result.select("id", F.explode("language").alias("l")).select(
    "id",
    F.col("l.metadata")["sentence"].cast("int").alias("n"),
    F.col("l.result").alias("language"),
    F.col("l.metadata")["confidence"].alias("confidence"),
)
sentences = sentence_rows.join(language_rows, ["id", "n"], "left").orderBy("id", "n")
sentences.show(truncate=70)

## Keep the English text

Uncertain sentences (`unk`) are kept rather than dropped: they are mostly short lines such as lab values or abbreviations, but a short line in another Latin-script language (a two-word Spanish reply, say) becomes `unk` too and is kept. Change `keep` if you prefer to set them aside for review.

As a safety net, a sentence that contains letters of another writing system (Chinese, Japanese, Korean, Arabic, Cyrillic, Devanagari, Thai, Hebrew) is set aside even when its label is English. That happens when two languages end up in one sentence, or for a name written in its own script, which an English-only step would not handle. Greek is left out of the list because of terms like β-blocker.

In [ ]:
other_script = r"[\p{IsHan}\p{IsHiragana}\p{IsKatakana}\p{IsHangul}\p{IsArabic}\p{IsCyrillic}\p{IsDevanagari}\p{IsThai}\p{IsHebrew}]"
english_or_unsure = (F.col("language") == "eng_Latn") | (F.col("language") == "unk") | F.col("language").startswith("zxx_")
keep = english_or_unsure & ~F.col("sentence").rlike(other_script)

english_only = sentences.where(keep) \
    .groupBy("id") \
    .agg(F.sort_array(F.collect_list(F.struct("n", "sentence"))).alias("kept")) \
    .select("id", F.concat_ws(" ", "kept.sentence").alias("english_text")) \
    .orderBy("id")
english_only.show(truncate=100)

sentences.where(~keep).show(truncate=70)

## Measure it on your own labeled data

`Benchmark.evaluate` scores the pipeline against gold labels (one label per row), and `Benchmark.throughput` measures speed. The gold labels use GlotLID's codes. On a handful of rows the throughput is mostly Spark's per-job overhead; measure it on a few thousand rows of your own data (one core handles about 5,000 sentences a second).

In [ ]:
from sparknlp.benchmark import Benchmark

gold = spark.createDataFrame([
    ("The wound is clean and dry, with no signs of infection.", "eng_Latn"),
    ("La herida está limpia y seca, sin signos de infección.", "spa_Latn"),
    ("伤口清洁干燥,没有感染迹象。", "cmn_Hani"),
    ("Vết thương sạch và khô, không có dấu hiệu nhiễm trùng.", "vie_Latn"),
    ("Ang sugat ay malinis at tuyo, walang palatandaan ng impeksyon.", "tgl_Latn"),
    ("상처는 깨끗하고 건조하며 감염 징후가 없습니다.", "kor_Hang"),
], ["text", "label"])

document_language = LanguageDetectorDL.pretrained("ld_glotlid_v4", "xx") \
    .setInputCols(["document"]) \
    .setOutputCol("language")
benchmark_pipeline = Pipeline(stages=[document_assembler, document_language]).fit(gold)
print(Benchmark.evaluate(benchmark_pipeline, gold, task="languagedetection", label_col="label"))
print(Benchmark.throughput(benchmark_pipeline, gold))